# Track 1 · 02 Features — run REVE once, cache what the head will fit on

**One job.** Turn the EEG that `track1_01_prepare` put on Drive into cached encoder features,
paired with their target embeddings. The encoder runs exactly once. Everything after
this is a cheap head fit you can redo as often as you like.

Self-contained: upload this one file, no clone.

---

## What this caches, and why two forms

REVE's tokens come out as `(batch, n_chans, n_patches, embed_dim)`. With `patch_size=200`
and a 1.0 s window at 200 Hz there is exactly **one time patch**, so the 32 tokens *are*
the 32 electrodes. Verified against the checkpoint:

```
config                     features             flatten  pooled
32 ch, 200 samp (ours)     (B, 32, 1, 512)       16,384     512
16 ch, 200 samp            (B, 16, 1, 512)        8,192     512
32 ch, 400 samp            (B, 32, 2, 512)       32,768     512
```

So two things get written:

| | where | size | why |
|---|---|---|---|
| pooled `(n, 512)` | Drive | ~52 MB | what the rung-1 ridge fits, survives a session reset |
| tokens `(n, 32, 512)` fp16 | `/content` | ~840 MB | keeps the better head reachable without re-encoding |

**Why pooled at all.** REVE's `final_layer` flattens `n_chans x n_patches x embed_dim`, so
a head fitted at 16,384 raises `mat1 and mat2 shapes cannot be multiplied (2x8192 and
16384x1536)` as soon as the geometry changes. The corpora in play disagree on channel
count (nm000134 is 32 ch, nm000133 is 64 ch, the warm-up board scores on THINGS-EEG2), so
a flatten head is not transferable. Pooling to `embed_dim` is what makes the head portable.
This is the same reason the solver drops `final_layer`.

**Why tokens too.** Mean-pooling averages across *electrodes*, and on a visual task the
signal is occipital, so this is the main thing limiting rung 1. The fix is to learn the
pooling: `attention_pooling=True` uses a `cls_query_token`, which is **absent from
`brain-bzh/reve-base`** (loading reports exactly `['cls_query_token']` missing beyond the
four deliberate `final_layer` omissions). Training that one 512-parameter tensor alongside
the 512 -> 1536 linear gives channel-count invariance *and* learned channel weighting. With
tokens cached, that costs no second encoder pass.

## Preprocessing is specified, not chosen

From the braindecode model card: input **must be 200 Hz** to match pretraining, and the
authors apply **z-score then clipping at 15 SD**. Epoching comes from the track config:
`start: -0.2`, `duration: 1.0`, baseline over the first 0.2 s.

One genuine unknown: the card does not say whether the z-score is per channel or over the
whole window. `ZSCORE_SCOPE` below defaults to `channel`, which is the usual EEG choice.
It is worth revisiting if rung 1 disappoints.

No montage is needed. REVE resolves electrode positions from channel *names* through its
own position bank, so `chs_info=[{'ch_name': ...}]` is sufficient; the `montage present`
warning in `track1_01_prepare` does not apply to this encoder.

## Setup

In [ ]:
ROOT = '/content/drive/MyDrive/Neurips26'

SFREQ = 200.0              # REVE is pretrained at 200 Hz; not negotiable
TMIN, DURATION = -0.2, 1.0 # from neuralbench/tasks/eeg/image/config.yaml
BASELINE = (-0.2, 0.0)
CLIP_SD = 15.0             # from the REVE model card
ZSCORE_SCOPE = 'channel'   # 'channel' or 'window'

ENCODER = 'brain-bzh/reve-base'
BATCH = 128                # windows per forward pass
KEEP_TOKENS = True         # also cache (n, 32, 512) for the learned-pooling head
TOKEN_DIR = '/content/tokens'   # ephemeral on purpose; ~840 MB, regenerable

N_TIMES = int(round(DURATION * SFREQ))
TMAX = TMIN + DURATION - 1.0 / SFREQ     # inclusive endpoint -> exactly N_TIMES samples
print(f'window  {TMIN} to {TMAX:.3f} s at {SFREQ:g} Hz  ->  {N_TIMES} samples')
print(f'encoder {ENCODER}')
print(f'tokens  {"cached to " + TOKEN_DIR if KEEP_TOKENS else "not cached"}')

In [ ]:
%pip install -q mne 'braindecode>=1.8' huggingface_hub safetensors

import importlib
for mod in ('mne', 'braindecode', 'torch', 'numpy', 'pandas'):
    m = importlib.import_module(mod)
    print(f'{mod:14} {getattr(m, "__version__", "ok")}')

In [ ]:
import os, json, time, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np, pandas as pd, torch, mne
mne.set_log_level('ERROR')

DATASET, SLUG = 'nm000134', 'Track1'
STIM_KINDS = {'train': 'stim_train', 'test': 'stim_test'}

if ROOT.startswith('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

ROOT = Path(ROOT)
BASE = ROOT / SLUG
RAW, PREP = BASE / 'raw', BASE / 'prepared'
FEAT = BASE / 'features'
FEAT.mkdir(parents=True, exist_ok=True)
TOKENS = Path(TOKEN_DIR)
if KEEP_TOKENS:
    TOKENS.mkdir(parents=True, exist_ok=True)

os.environ['HF_HOME'] = '/content/hf'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'


def report(title, checks):
    print(title)
    for name, ok, detail in checks:
        print(f"  [{'ok  ' if ok else 'FAIL'}] {name}: {detail}")
    print(f'  -> {sum(c[1] for c in checks)} ok, {sum(not c[1] for c in checks)} failed')


print(f'features -> {FEAT}')
print(f'device      {DEVICE}')
if DEVICE == 'cpu':
    print('\nNo GPU. A 69M-parameter transformer over ~25,000 windows will be slow.')

## Step 1 — load the target table and the recording list

Both come from `track1_01_prepare`. The manifest tells us which recordings were used and how they
were split, so this notebook never needs to query EEGDash.

The table is checked before anything is built on it. Coverage in `track1_01_prepare` proved every
image produced a row; this checks the rows are usable.

In [ ]:
z = np.load(PREP / 'image_embeddings.npz', allow_pickle=True)
keys, emb = z['keys'], z['emb'].astype('float32')
manifest = json.loads((PREP / 'prepare_manifest.json').read_text())

norms = np.linalg.norm(emb, axis=1)
report('target table:', [
    ('loaded', True, f'{emb.shape} {emb.dtype} from {str(z["model"])}'),
    ('finite values', bool(np.isfinite(emb).all()), 'no NaN or Inf'),
    ('distinct keys', len(keys) == len(set(keys.tolist())), f'{len(keys):,} keys'),
    ('no zero rows', bool((norms > 0).all()), f'{int((norms == 0).sum())} all-zero'),
    ('dim 1536', emb.shape[1] == 1536, f'{emb.shape[1]}'),
    ('norms', bool(norms.max() / max(norms.min(), 1e-9) < 10),
     f'min {norms.min():.2f}  median {np.median(norms):.2f}  max {norms.max():.2f}'),
])

key_to_idx = {k: i for i, k in enumerate(keys.tolist())}
recordings = manifest['recordings']
print(f'\n{len(recordings)} recordings  '
      f'({sum(v["split"] == "train" for v in recordings.values())} train, '
      f'{sum(v["split"] == "test" for v in recordings.values())} test)')
print(f'expected windows: ~{sum(v["events"] for v in recordings.values()):,}')

## Step 2 — build the encoder

Loaded from the Hub with our dimensions. The 139 encoder tensors are bit-identical to the
`weights.pt` the Track 1 submission ships, so the features cached here come from exactly
the encoder that will run at inference.

In [ ]:
from braindecode.models import REVE

first = sorted(recordings)[0]
probe_raw = mne.io.read_raw_edf(RAW / DATASET / first, preload=False)
CH_NAMES = probe_raw.ch_names
CHS_INFO = [{'ch_name': c} for c in CH_NAMES]
print(f'{len(CH_NAMES)} channels: {CH_NAMES[:6]} ...')

encoder = REVE.from_pretrained(
    ENCODER, n_outputs=1536, n_chans=len(CH_NAMES),
    n_times=N_TIMES, sfreq=SFREQ, chs_info=CHS_INFO).to(DEVICE).eval()

with torch.no_grad():
    probe = encoder(torch.randn(2, len(CH_NAMES), N_TIMES, device=DEVICE),
                    return_features=True)['features']
n_chans_out, n_patches, embed_dim = probe.shape[1:]
report('encoder:', [
    ('features', probe.ndim == 4, f'{tuple(probe.shape)} (batch, chan, patch, embed)'),
    ('one time patch', n_patches == 1, f'{n_patches} patch per channel at {N_TIMES} samples'),
    ('tokens are channels', n_chans_out == len(CH_NAMES), f'{n_chans_out} tokens'),
    ('embed dim', embed_dim == 512, f'{embed_dim}'),
    ('finite', bool(torch.isfinite(probe).all()), 'no NaN or Inf'),
])
print(f'\nflatten would be {n_chans_out * n_patches * embed_dim:,} (corpus-specific); '
      f'pooled is {embed_dim} (portable)')
del probe

## Step 3 — epoch and encode, one recording at a time

Resumable: a recording whose outputs already exist is skipped, so a dropped session costs
one recording rather than the whole pass.

In [ ]:
def prepare_windows(rel, split):
    """Epoch one recording and return (X, stim_files). X is (n, chans, N_TIMES) float32."""
    raw = mne.io.read_raw_edf(RAW / DATASET / rel, preload=True)
    raw.resample(SFREQ)

    tsv = RAW / DATASET / rel.replace('_eeg.edf', '_events.tsv')
    df = pd.read_csv(tsv, sep='\t')
    df = df[df.trial_type == STIM_KINDS[split]].dropna(subset=['stim_file']).reset_index(drop=True)

    samples = np.round(df.onset.values * SFREQ).astype(int)
    lo, hi = int(round(TMIN * SFREQ)), int(round(TMAX * SFREQ))
    keep = (samples + lo >= 0) & (samples + hi < raw.n_times)
    df, samples = df[keep].reset_index(drop=True), samples[keep]

    events = np.column_stack([samples, np.zeros_like(samples), np.ones_like(samples)])
    epochs = mne.Epochs(raw, events, tmin=TMIN, tmax=TMAX, baseline=BASELINE,
                        preload=True, reject=None, flat=None, verbose=False)
    X = epochs.get_data(copy=True).astype('float32')
    stim = df.stim_file.values[epochs.selection]      # epochs may drop events; stay aligned

    axis = 2 if ZSCORE_SCOPE == 'channel' else (1, 2)
    mu = X.mean(axis=axis, keepdims=True)
    sd = X.std(axis=axis, keepdims=True)
    X = np.clip((X - mu) / (sd + 1e-8), -CLIP_SD, CLIP_SD)
    return X, stim


@torch.no_grad()
def encode(X):
    """(n, chans, times) -> pooled (n, 512) float32 and tokens (n, chans, 512) float16."""
    pooled, tokens = [], []
    for i in range(0, len(X), BATCH):
        batch = torch.from_numpy(X[i:i + BATCH]).to(DEVICE)
        f = encoder(batch, return_features=True)['features']   # (b, chan, patch, embed)
        pooled.append(f.mean(dim=(1, 2)).float().cpu().numpy())
        if KEEP_TOKENS:
            tokens.append(f.mean(dim=2).half().cpu().numpy())   # collapse the single patch
    return np.concatenate(pooled), (np.concatenate(tokens) if KEEP_TOKENS else None)

In [ ]:
t0 = time.time()
done, skipped, total_windows, unknown = 0, 0, 0, 0

for rel in sorted(recordings):
    split = recordings[rel]['split']
    stem = rel.replace('/', '__').replace('_eeg.edf', '')
    out = FEAT / f'{stem}.npz'
    tok = TOKENS / f'{stem}.npy'
    if out.exists() and (not KEEP_TOKENS or tok.exists()):
        skipped += 1
        total_windows += int(np.load(out)['pooled'].shape[0])
        continue

    X, stim = prepare_windows(rel, split)
    idx = np.array([key_to_idx.get(s, -1) for s in stim], dtype='int32')
    unknown += int((idx < 0).sum())
    pooled, tokens = encode(X)

    np.savez(out, pooled=pooled.astype('float32'), image_idx=idx,
             stim_file=np.array(stim), split=split,
             subject=recordings[rel]['subject'], rel=rel)
    if KEEP_TOKENS:
        np.save(tok, tokens)

    done += 1
    total_windows += len(pooled)
    print(f'  {stem:<44} {split:<5} {len(pooled):5,} windows  '
          f'{time.time() - t0:5.0f}s elapsed')

print(f'\n{done} encoded, {skipped} reused, {total_windows:,} windows in '
      f'{(time.time() - t0) / 60:.1f} min')
if unknown:
    print(f'WARNING: {unknown} windows had a stim_file absent from the target table')

## Step 4 — consolidate and check

In [ ]:
parts = {'train': [], 'test': []}
dropped = 0
for f in sorted(FEAT.glob('*.npz')):
    d = np.load(f, allow_pickle=True)
    idx = d['image_idx']
    # A window whose stim_file is not in the table would index emb[-1] and quietly
    # train against the wrong image, so drop it here rather than let it propagate.
    keep = idx >= 0
    dropped += int((~keep).sum())
    parts[str(d['split'])].append(
        (d['pooled'][keep], idx[keep],
         np.full(int(keep.sum()), str(d['subject']))))
if dropped:
    print(f'dropped {dropped:,} windows with no target embedding\n')

out = {}
for split, items in parts.items():
    out[f'X_{split}'] = np.concatenate([a for a, _, _ in items]).astype('float32')
    out[f'y_{split}'] = np.concatenate([b for _, b, _ in items]).astype('int32')
    out[f'subj_{split}'] = np.concatenate([c for _, _, c in items])

# The target table is NOT bundled in here. It already lives in image_embeddings.npz,
# and duplicating 58 MB of it onto Drive buys nothing; y_* are indices into it.
bundle = PREP / 'features_pooled.npz'
np.savez(bundle, encoder=ENCODER, sfreq=SFREQ, zscore_scope=ZSCORE_SCOPE,
         table='image_embeddings.npz', **out)

Xtr, Xte = out['X_train'], out['X_test']
ng = len(set(out['y_test'].tolist()))
spread = Xtr.std(axis=0)
report('features:', [
    ('train', Xtr.ndim == 2 and Xtr.shape[1] == 512, f'{Xtr.shape}'),
    ('test', Xte.ndim == 2 and Xte.shape[1] == 512, f'{Xte.shape}'),
    ('finite', bool(np.isfinite(Xtr).all() and np.isfinite(Xte).all()), 'no NaN or Inf'),
    ('targets resolved', bool((out['y_train'] >= 0).all() and (out['y_test'] >= 0).all()),
     f'{int((out["y_train"] < 0).sum() + (out["y_test"] < 0).sum())} unresolved'),
    ('not collapsed', bool(spread.min() > 1e-6),
     f'per-dim std across windows: min {spread.min():.2e} median {np.median(spread):.2e}'),
    ('test gallery', ng > 1,
     f'{ng} distinct test images, '
     f'chance top-5 {min(5, ng) / max(ng, 1):.2%}'),
    ('subjects', True, f'{len(set(out["subj_train"]))} train / {len(set(out["subj_test"]))} test'),
])

print(f'\n{bundle}  {bundle.stat().st_size / 1e6:.0f} MB')
if KEEP_TOKENS:
    tb = sum(f.stat().st_size for f in TOKENS.glob("*.npy")) / 1e6
    print(f'{TOKENS}  {tb:.0f} MB  ({len(list(TOKENS.glob("*.npy")))} files, ephemeral)')
print()
for label, path in (('EEG', RAW), ('target table', PREP / 'image_embeddings.npz'),
                    ('feature shards', FEAT), ('feature bundle', bundle),
                    ('embedding shards', PREP / 'embedding_parts')):
    if not path.exists():
        continue
    mb = (path.stat().st_size if path.is_file()
          else sum(f.stat().st_size for f in path.rglob('*') if f.is_file())) / 1e6
    print(f'  {label:<17}{mb:6.0f} MB')
print(f'  {"Drive total":<17}'
      f'{sum(f.stat().st_size for f in BASE.rglob("*") if f.is_file()) / 1e6:6.0f} MB')
print('\nembedding_parts is only needed to resume an interrupted T1 run; once the')
print('table checks out it can be deleted.')
print('\nNext: fit the 512 -> 1536 head by closed-form ridge against L2-normalised targets,')
print('then score top-5 retrieval against the test gallery.')